This file is responsible for fetching latest data from the Binance API

In [4]:
# Imports

import numpy as np
import pandas as pd
import binance.client as client
import requests
import time
import math

print("All libraries imported successfully.")

All libraries imported successfully.


In [11]:
# Constants

TRADING_PAIRS = []


In [2]:
# Pair Selection

while True:
    TRADING_PAIRS = input("Input the trading pairs (comma-separated, e.g. BTCUSDT,ETHUSDT) or 0 for default: ").strip().upper().split(",")
    
    if TRADING_PAIRS[0] == "0":
        TRADING_PAIRS = ["ETHUSDT"]
        break
    if len(TRADING_PAIRS) >  2 :
        print("Invalid trading pair (too many). Please try again.")
        continue
    for i in range(len(TRADING_PAIRS)):
        if len(TRADING_PAIRS[i]) < 6:
            print(f"Invalid trading pair: {TRADING_PAIRS[i]}. Please try again.")
            continue
    if(len(TRADING_PAIRS) <0):
        print("No trading pairs available. Please try again.")
        continue
    break

print(f"Using trading pairs: {TRADING_PAIRS}")

Using trading pairs: ['ETHUSDT', 'BTCUSDT']


In [13]:
# Data Fetching
# Download historical depth range using helper module

import importlib
import book_depth_utils
importlib.reload(book_depth_utils)

period_input = input("Enter period (e.g. 7d, 1w, 1m, 1y, or YYYY-MM-DD/YYYY-MM-DD): ").strip()
if not period_input:
    period_input = "7d"

summary = book_depth_utils.download_book_depth_range(TRADING_PAIRS, period_input, out_base="bookDepth_data", pause_seconds=0.15)
print("Download summary:")
print(f"Symbols: {summary['requested_symbols']}")
print(f"Period: {summary['start_date']} to {summary['end_date']} ({summary['days_requested']} days)")
print(f"Downloaded: {summary['downloaded']}")
print(f"Missing (404): {summary['skipped_404']}")
print(f"Errors: {summary['errors']}")
print("Detail sample:")
for line in summary["details"][:10]:
    print(" -", line)



Download summary:
Symbols: ['ETHUSDT', 'BTCUSDT']
Period: 2026-03-15 to 2026-03-21 (7 days)
Downloaded: 12
Missing (404): 2
Errors: 0
Detail sample:
 - ETHUSDT 2026-03-15: downloaded
 - ETHUSDT 2026-03-16: downloaded
 - ETHUSDT 2026-03-17: downloaded
 - ETHUSDT 2026-03-18: downloaded
 - ETHUSDT 2026-03-19: downloaded
 - ETHUSDT 2026-03-20: downloaded
 - ETHUSDT 2026-03-21: missing (not available)
 - BTCUSDT 2026-03-15: downloaded
 - BTCUSDT 2026-03-16: downloaded
 - BTCUSDT 2026-03-17: downloaded


In [5]:
# Merge and clean data
import glob
from pathlib import Path
for pair in TRADING_PAIRS:
    path = Path("bookDepth_data"+"/"+pair)
    if(path.is_dir()):
        all_files = glob.glob(str(path / "*.csv"))
        df_list = []
        for file in all_files:
            df = pd.read_csv(file)
            df_list.append(df)
        if df_list:
            merged_df = pd.concat(df_list, ignore_index=True)
            merged_df.to_csv(f"bookDepth_data/{pair}/{pair}_merged.csv", index=False)
            print(f"Merged {len(df_list)} files for {pair} into {pair}_merged.csv")
        else:
            print(f"No CSV files found for {pair} in {path}")


Merged 6 files for ETHUSDT into ETHUSDT_merged.csv
Merged 6 files for BTCUSDT into BTCUSDT_merged.csv


In [ ]:
# Download historical trades using helper module

import importlib
import trades_utils
importlib.reload(trades_utils)

summary = trades_utils.download_trades_range(TRADING_PAIRS, period_input, out_base="trades_data", pause_seconds=0.15)
print("Trade download summary:")
print(f"Symbols: {summary['requested_symbols']}")
print(f"Period: {summary['start_date']} to {summary['end_date']} ({summary['days_requested']} days)")
print(f"Downloaded: {summary['downloaded']}")
print(f"Missing (404): {summary['skipped_404']}")
print(f"Errors: {summary['errors']}")
for line in summary["details"][:10]:
    print(" -", line)


Trade download summary:
Symbols: ['ETHUSDT', 'BTCUSDT']
Period: 2026-03-15 to 2026-03-21 (7 days)
Downloaded: 12
Missing (404): 2
Errors: 0
 - ETHUSDT 2026-03-15: downloaded
 - ETHUSDT 2026-03-16: downloaded
 - ETHUSDT 2026-03-17: downloaded
 - ETHUSDT 2026-03-18: downloaded
 - ETHUSDT 2026-03-19: downloaded
 - ETHUSDT 2026-03-20: downloaded
 - ETHUSDT 2026-03-21: missing (not available)
 - BTCUSDT 2026-03-15: downloaded
 - BTCUSDT 2026-03-16: downloaded
 - BTCUSDT 2026-03-17: downloaded


In [8]:
# Merge daily trade CSV files per pair into one file
import glob
from pathlib import Path
import pandas as pd

TRADE_DTYPES = {
    "id": "int64",
    "price": "float64",
    "qty": "float64",
    "quote_qty": "float64",
    "time": "int64",
    "is_buyer_maker": "bool",
}

for pair in TRADING_PAIRS:
    path = Path("trades_data") / pair
    if path.is_dir():
        all_files = sorted(glob.glob(str(path / f"{pair}-trades-*.csv")))
        df_list = []
        for file in all_files:
            df = pd.read_csv(file, dtype=TRADE_DTYPES)
            df_list.append(df)
        if df_list:
            merged_df = pd.concat(df_list, ignore_index=True)
            out_path = path / f"{pair}_trades_merged.csv"
            merged_df.to_csv(out_path, index=False)
            print(f"Merged {len(df_list)} files for {pair}: {len(merged_df)} rows → {out_path}")
        else:
            print(f"No trade CSV files found for {pair} in {path}")


Merged 6 files for ETHUSDT: 50881839 rows → trades_data\ETHUSDT\ETHUSDT_trades_merged.csv
Merged 6 files for BTCUSDT: 30609989 rows → trades_data\BTCUSDT\BTCUSDT_trades_merged.csv
